In [1]:
#pip install mediapipe==0.10.21 opencv-python numpy scipy

import time
from collections import deque

import cv2
import mediapipe as mp
import numpy as np
from scipy.signal import butter, filtfilt, detrend


# =========================
# 설정값
# =========================

CAMERA_INDEX = 0

MAX_BUFFER_SEC = 40

HR_WINDOW_SEC = 20          # 심박수 계산용 시간창
RESP_WINDOW_SEC = 30        # 호흡수 계산용 시간창

HR_BAND = (0.75, 3.0)       # 45~180 bpm
RESP_BAND = (0.10, 0.50)    # 6~30 rpm

RESP_SIZE = (80, 80)        # EVM용 ROI 크기
RESP_ALPHA = 20.0           # EVM 증폭 계수

ESTIMATE_INTERVAL = 1.0     # 초 단위로 결과 갱신


# =========================
# 신호 처리 함수
# =========================

def bandpass_filter(x, fs, low, high, order=3, axis=0):
    nyq = fs * 0.5
    high = min(high, nyq * 0.95)

    if low >= high:
        return None

    b, a = butter(order, [low / nyq, high / nyq], btype="band")

    n = x.shape[axis]
    padlen = min(3 * (max(len(a), len(b)) - 1), n - 1)

    if padlen < 1:
        return None

    return filtfilt(b, a, x, axis=axis, padlen=padlen)


def dominant_rate_from_signal(signal, fs, low_hz, high_hz):
    signal = np.asarray(signal, dtype=np.float32)

    if len(signal) < int(fs * 5):
        return None, None

    signal = detrend(signal)
    signal = signal - np.mean(signal)

    window = np.hanning(len(signal))
    spectrum = np.abs(np.fft.rfft(signal * window))
    freqs = np.fft.rfftfreq(len(signal), d=1.0 / fs)

    mask = (freqs >= low_hz) & (freqs <= high_hz)

    if not np.any(mask):
        return None, None

    band_spectrum = spectrum[mask]
    band_freqs = freqs[mask]

    peak_idx = np.argmax(band_spectrum)
    peak_freq = band_freqs[peak_idx]

    confidence = band_spectrum[peak_idx] / (np.median(band_spectrum) + 1e-8)

    return peak_freq * 60.0, confidence


def resample_rgb_signal(times, rgb_values, target_fs=30.0):
    times = np.asarray(times, dtype=np.float64)
    values = np.asarray(rgb_values, dtype=np.float32)

    if len(times) < 10:
        return None, None

    t0, t1 = times[0], times[-1]
    duration = t1 - t0

    if duration < 5:
        return None, None

    uniform_t = np.arange(t0, t1, 1.0 / target_fs)

    if len(uniform_t) < 10:
        return None, None

    out = np.zeros((len(uniform_t), 3), dtype=np.float32)

    for c in range(3):
        out[:, c] = np.interp(uniform_t, times, values[:, c])

    return out, target_fs


def estimate_heart_rate_pos(times, rgb_values):
    times = np.asarray(times, dtype=np.float64)
    rgb_values = np.asarray(rgb_values, dtype=np.float32)

    if len(times) < 30:
        return None, None

    latest_t = times[-1]
    mask = times >= latest_t - HR_WINDOW_SEC

    times = times[mask]
    rgb_values = rgb_values[mask]

    if len(times) < 30 or times[-1] - times[0] < 10:
        return None, None

    rgb, fs = resample_rgb_signal(times, rgb_values, target_fs=30.0)

    if rgb is None:
        return None, None

    rgb = rgb / (np.mean(rgb, axis=0) + 1e-6) - 1.0
    rgb = detrend(rgb, axis=0)

    r = rgb[:, 0]
    g = rgb[:, 1]
    b = rgb[:, 2]

    s1 = g - b
    s2 = g + b - 2 * r

    alpha = np.std(s1) / (np.std(s2) + 1e-8)
    pulse_signal = s1 + alpha * s2

    filtered = bandpass_filter(
        pulse_signal,
        fs,
        HR_BAND[0],
        HR_BAND[1],
        order=3,
        axis=0
    )

    if filtered is None:
        return None, None

    bpm, confidence = dominant_rate_from_signal(
        filtered,
        fs,
        HR_BAND[0],
        HR_BAND[1]
    )

    return bpm, confidence


def estimate_respiration_evm(times, frames):
    times = np.asarray(times, dtype=np.float64)

    if len(times) < 30:
        return None, None

    latest_t = times[-1]
    mask = times >= latest_t - RESP_WINDOW_SEC

    selected_times = times[mask]
    selected_frames = [frame for frame, keep in zip(frames, mask) if keep]

    if len(selected_frames) < 30:
        return None, None

    duration = selected_times[-1] - selected_times[0]

    if duration < 15:
        return None, None

    fs = (len(selected_frames) - 1) / duration

    if fs <= 2:
        return None, None

    stack = np.asarray(selected_frames, dtype=np.float32)

    mean_frame = np.mean(stack, axis=0)
    normalized = stack / (mean_frame + 1e-6) - 1.0

    filtered = bandpass_filter(
        normalized,
        fs,
        RESP_BAND[0],
        RESP_BAND[1],
        order=2,
        axis=0
    )

    if filtered is None:
        return None, None

    magnified = filtered * RESP_ALPHA

    resp_signal = np.mean(magnified, axis=(1, 2))

    rpm, confidence = dominant_rate_from_signal(
        resp_signal,
        fs,
        RESP_BAND[0],
        RESP_BAND[1]
    )

    return rpm, confidence


# =========================
# ROI 처리 함수
# =========================

def clamp_rect(rect, width, height):
    x1, y1, x2, y2 = rect

    x1 = int(max(0, min(width - 1, x1)))
    y1 = int(max(0, min(height - 1, y1)))
    x2 = int(max(0, min(width - 1, x2)))
    y2 = int(max(0, min(height - 1, y2)))

    if x2 <= x1 + 5 or y2 <= y1 + 5:
        return None

    return x1, y1, x2, y2


def get_face_bbox(landmarks, frame_width, frame_height):
    xs = np.array([lm.x for lm in landmarks]) * frame_width
    ys = np.array([lm.y for lm in landmarks]) * frame_height

    x1, y1 = np.min(xs), np.min(ys)
    x2, y2 = np.max(xs), np.max(ys)

    return x1, y1, x2, y2


def build_rois(landmarks, frame_width, frame_height):
    x1, y1, x2, y2 = get_face_bbox(landmarks, frame_width, frame_height)

    fw = x2 - x1
    fh = y2 - y1

    if fw <= 0 or fh <= 0:
        return [], None, None

    face_rect = clamp_rect((x1, y1, x2, y2), frame_width, frame_height)

    # rPPG용 ROI: 이마 + 양 볼
    forehead = clamp_rect(
        (
            x1 + 0.25 * fw,
            y1 + 0.10 * fh,
            x1 + 0.75 * fw,
            y1 + 0.28 * fh,
        ),
        frame_width,
        frame_height
    )

    left_cheek = clamp_rect(
        (
            x1 + 0.15 * fw,
            y1 + 0.45 * fh,
            x1 + 0.38 * fw,
            y1 + 0.68 * fh,
        ),
        frame_width,
        frame_height
    )

    right_cheek = clamp_rect(
        (
            x1 + 0.62 * fw,
            y1 + 0.45 * fh,
            x1 + 0.85 * fw,
            y1 + 0.68 * fh,
        ),
        frame_width,
        frame_height
    )

    pulse_rois = [r for r in [forehead, left_cheek, right_cheek] if r is not None]

    # 호흡 EVM용 ROI
    chest_roi = clamp_rect(
        (
            x1 - 0.25 * fw,
            y2 + 0.05 * fh,
            x2 + 0.25 * fw,
            y2 + 0.95 * fh,
        ),
        frame_width,
        frame_height
    )

    if chest_roi is None:
        chest_roi = clamp_rect(
            (
                x1 + 0.20 * fw,
                y1 + 0.55 * fh,
                x2 - 0.20 * fw,
                y1 + 0.90 * fh,
            ),
            frame_width,
            frame_height
        )

    return pulse_rois, chest_roi, face_rect


def mean_rgb_from_rois(frame_bgr, rois):
    values = []

    for rect in rois:
        x1, y1, x2, y2 = rect
        roi = frame_bgr[y1:y2, x1:x2]

        if roi.size == 0:
            continue

        rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)
        values.append(np.mean(rgb.reshape(-1, 3), axis=0))

    if not values:
        return None

    return np.mean(values, axis=0)


def get_resp_frame(frame_bgr, rect):
    x1, y1, x2, y2 = rect
    roi = frame_bgr[y1:y2, x1:x2]

    if roi.size == 0:
        return None

    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    small = cv2.resize(gray, RESP_SIZE, interpolation=cv2.INTER_AREA)
    small = small.astype(np.float32) / 255.0

    return small


def draw_rects(frame, rects, color, label=None):
    for rect in rects:
        x1, y1, x2, y2 = rect
        cv2.rectangle(frame, (x1, y1), (x2, y2), color, 1)

        if label:
            cv2.putText(
                frame,
                label,
                (x1, max(20, y1 - 5)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.45,
                color,
                1
            )


def draw_landmark_points(frame, landmarks, width, height, color=(0, 255, 255), radius=1):
    for lm in landmarks:
        x = int(lm.x * width)
        y = int(lm.y * height)

        if 0 <= x < width and 0 <= y < height:
            cv2.circle(frame, (x, y), radius, color, -1)


def ema(old, new, alpha=0.25):
    if new is None:
        return old

    if old is None:
        return new

    return old * (1.0 - alpha) + new * alpha


# =========================
# 메인 실행
# =========================

def main():
    mp_face_mesh = mp.solutions.face_mesh

    cap = cv2.VideoCapture(CAMERA_INDEX)

    if not cap.isOpened():
        print("웹캠을 열 수 없습니다.")
        return

    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    pulse_times = deque()
    pulse_rgbs = deque()

    resp_times = deque()
    resp_frames = deque()

    bpm_value = None
    rpm_value = None
    bpm_conf = None
    rpm_conf = None

    last_estimate_time = 0.0

    with mp_face_mesh.FaceMesh(
        max_num_faces=1,
        refine_landmarks=True,
        min_detection_confidence=0.6,
        min_tracking_confidence=0.6
    ) as face_mesh:

        while True:
            ret, frame = cap.read()

            if not ret:
                print("프레임을 읽을 수 없습니다.")
                break

            frame = cv2.flip(frame, 1)
            height, width = frame.shape[:2]

            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = face_mesh.process(rgb_frame)

            now = time.perf_counter()
            face_detected = False

            if results.multi_face_landmarks:
                face_detected = True

                face_landmarks = results.multi_face_landmarks[0]
                landmarks = face_landmarks.landmark

                # 얼굴 랜드마크 점만 표시
                draw_landmark_points(frame, landmarks, width, height, color=(0, 255, 255), radius=1)

                pulse_rois, resp_roi, face_rect = build_rois(
                    landmarks,
                    width,
                    height
                )

                if face_rect is not None:
                    draw_rects(frame, [face_rect], (255, 255, 255), "FACE")

                if pulse_rois:
                    mean_rgb = mean_rgb_from_rois(frame, pulse_rois)

                    if mean_rgb is not None:
                        pulse_times.append(now)
                        pulse_rgbs.append(mean_rgb)

                    draw_rects(frame, pulse_rois, (0, 255, 0), "rPPG")

                if resp_roi is not None:
                    resp_frame = get_resp_frame(frame, resp_roi)

                    if resp_frame is not None:
                        resp_times.append(now)
                        resp_frames.append(resp_frame)

                    draw_rects(frame, [resp_roi], (255, 0, 0), "EVM RESP")

            # 오래된 버퍼 제거
            while pulse_times and now - pulse_times[0] > MAX_BUFFER_SEC:
                pulse_times.popleft()
                pulse_rgbs.popleft()

            while resp_times and now - resp_times[0] > MAX_BUFFER_SEC:
                resp_times.popleft()
                resp_frames.popleft()

            # 결과 갱신
            if now - last_estimate_time >= ESTIMATE_INTERVAL:
                last_estimate_time = now

                if len(pulse_times) > 0:
                    bpm, conf = estimate_heart_rate_pos(
                        list(pulse_times),
                        list(pulse_rgbs)
                    )

                    if bpm is not None and 40 <= bpm <= 200:
                        bpm_value = ema(bpm_value, bpm)
                        bpm_conf = conf

                if len(resp_times) > 0:
                    rpm, conf = estimate_respiration_evm(
                        list(resp_times),
                        list(resp_frames)
                    )

                    if rpm is not None and 4 <= rpm <= 40:
                        rpm_value = ema(rpm_value, rpm)
                        rpm_conf = conf

            # 화면 출력
            if face_detected:
                status = "Face detected"
            else:
                status = "No face"

            cv2.putText(
                frame,
                status,
                (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (255, 255, 255),
                2
            )

            if bpm_value is None:
                hr_text = f"Heart Rate: measuring... {len(pulse_times):d} frames"
            else:
                hr_text = f"Heart Rate: {bpm_value:.1f} BPM"

                if bpm_conf is not None:
                    hr_text += f"  conf:{bpm_conf:.1f}"

            if rpm_value is None:
                resp_text = f"Respiration: measuring... {len(resp_times):d} frames"
            else:
                resp_text = f"Respiration: {rpm_value:.1f} RPM"

                if rpm_conf is not None:
                    resp_text += f"  conf:{rpm_conf:.1f}"

            cv2.putText(
                frame,
                hr_text,
                (20, 65),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (0, 255, 0),
                2
            )

            cv2.putText(
                frame,
                resp_text,
                (20, 100),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7,
                (255, 0, 0),
                2
            )

            cv2.putText(
                frame,
                "Press q to quit",
                (20, height - 20),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 255),
                1
            )

            cv2.imshow("Real-time rPPG + EVM Respiration", frame)

            key = cv2.waitKey(1) & 0xFF

            if key == ord("q"):
                break

    cap.release()
    cv2.destroyAllWindows()


if __name__ == "__main__":
    main()